**The Laplacian matrix and graph partitioning.**
Given an undirected graph $G$ with $n$ nodes, the **Laplacian matrix** is:

$$\mathbf{L} = \mathbf{Deg} - \mathbf{Adj}$$

where $\mathbf{Adj}$ is the adjacency matrix ($\mathbf{Adj}_{ij}=1$ if nodes $i$ and $j$
are connected, 0 otherwise) and $\mathbf{Deg}$ is the diagonal degree matrix
($\mathbf{Deg}_{ii}$ = number of edges at node $i$).

The Laplacian is real and symmetric, so `np.linalg.eigh` is used to diagonalize it,
guaranteeing real eigenvalues in ascending order.
Key properties: the smallest eigenvalue is always 0 (corresponding to the constant
eigenvector), and the number of zero eigenvalues equals the number of connected
components in the graph.

In [ ]:
"""laplacian_matrix.ipynb"""

# Cell 01 - Build and draw a connected six-node undirected graph

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import display

from qis101_utils import as_latex

g = nx.Graph()
g.add_edge(0, 1)
g.add_edge(0, 2)
g.add_edge(1, 2)
g.add_edge(1, 3)
g.add_edge(2, 3)
g.add_edge(3, 4)
g.add_edge(4, 5)

plt.figure("laplacian_graph", figsize=(6, 5))
# fmt: off
nx.draw(g, nx.spring_layout(g, seed=1),
    node_size=500, node_color="skyblue",
    font_size=12, font_weight="bold",
    edge_color="gray", with_labels=True,
    arrows=True, arrowstyle="<->")
# fmt: on
plt.title("Graph Visualization")
plt.tight_layout()
plt.show()

**The adjacency matrix.**
$\mathbf{Adj}_{ij} = 1$ if there is an edge between nodes $i$ and $j$, otherwise 0.
For an undirected graph the matrix is symmetric: $\mathbf{Adj} = \mathbf{Adj}^T$.

In [ ]:
# Cell 02 - Adjacency matrix

a = nx.adjacency_matrix(g).toarray()
display(as_latex(a, prefix=r"\mathbf{Adj}="))

**The degree matrix.**
The degree of node $i$ is the number of edges connected to it,
equal to the sum of row $i$ (or column $i$) of the adjacency matrix.
For the symmetric adjacency matrix, `np.sum(a, axis=1)` (row sums)
and `np.sum(a, axis=0)` (column sums) are identical.
$\mathbf{Deg}$ places these degrees on the main diagonal.

In [ ]:
# Cell 03 - Degree matrix: row sums of adjacency matrix placed on the diagonal

s = np.sum(a, axis=1)  # row sums = degree of each node
d = np.diag(s)

display(as_latex(s, prefix=r"\mathrm{Row\;sums\;of\;}\mathbf{Adj}="))
display(as_latex(d, prefix=r"\mathbf{Deg}="))

**The Laplacian matrix.**
$\mathbf{L} = \mathbf{Deg} - \mathbf{Adj}$.
Each diagonal entry $L_{ii}$ is the degree of node $i$;
each off-diagonal entry $L_{ij} = -1$ if nodes $i$ and $j$ are connected, 0 otherwise.
Every row and column of $\mathbf{L}$ sums to zero.

In [ ]:
# Cell 04 - Laplacian = Degree - Adjacency

lap = d - a
display(as_latex(lap, prefix=r"\mathbf{Lap}="))

**Visualizing the Laplacian as a heatmap.**
The color map reveals the structure: positive diagonal entries (node degrees)
appear bright, negative off-diagonal entries (connected pairs) appear dark,
and zero entries (unconnected pairs) appear at the midpoint of the color scale.

In [ ]:
# Cell 05 - Heatmap of the Laplacian matrix

plt.figure("laplacian_heatmap", figsize=(5, 4))
plt.imshow(lap, cmap="viridis")
plt.title("Laplacian Matrix")
plt.colorbar()
plt.tight_layout()
plt.show()

**Eigendecomposition of the Laplacian.**
The Laplacian is real and symmetric, so `np.linalg.eigh` is used.
It returns real eigenvalues in **ascending order** and orthonormal eigenvectors,
with no manual sorting needed.
The smallest eigenvalue is always exactly 0 for a connected graph;
the corresponding eigenvector is the constant vector $[1,1,\ldots,1]/\sqrt{n}$.
The **second smallest** eigenvalue $\lambda_2$ (the Fiedler value) measures
how well-connected the graph is - larger means harder to cut.

In [ ]:
# Cell 06 - Eigenvalues and eigenvectors of the Laplacian
# eigh is correct here: Laplacian is real and symmetric
# It returns eigenvalues in ascending order - no sorting needed

eigen_vals, eigen_vecs = np.linalg.eigh(lap)

display(
    as_latex(eigen_vals, prefix=r"\mathrm{Eigenvalues\;(\lambda)\;of\;\mathbf{Lap}}=")
)
display(
    as_latex(
        eigen_vecs, prefix=r"\mathrm{Eigenvectors\;(\mathbf{v})\;of\;\mathbf{Lap}}="
    )
)

**The Fiedler vector.**
The **Fiedler vector** is the eigenvector corresponding to the *second smallest*
eigenvalue $\lambda_2$ of the Laplacian (column index 1 in the `eigh` output,
since eigenvalues are already sorted ascending).
It can be used to partition the graph into two subgraphs by the sign of each element:
nodes with non-negative entries form one partition, nodes with negative entries form the other.

This is widely used as a heuristic for the **sparsest cut** problem:
bipartition the nodes to minimize the ratio of edges crossing the cut
to the number of nodes in the smaller partition.
The Fiedler vector minimizes a relaxed (continuous) version of this objective
but does not guarantee the optimal cut - finding the true sparsest cut is NP-hard.

In [ ]:
# Cell 07 - The Fiedler vector: eigenvector of the 2nd smallest eigenvalue
# Since eigh returns eigenvalues ascending, column 1 is the Fiedler vector

f: np.ndarray = eigen_vecs[:, 1]
display(as_latex(f, prefix=r"\mathbf{F}="))

**Graph partitioning via the Fiedler vector.**
Nodes are assigned to partition P1 (non-negative Fiedler entries)
or P2 (negative Fiedler entries).
The number of edges crossing from P1 to P2 is the **cut size**;
the Fiedler partition minimizes a relaxed version of the cut-to-size ratio.

In [ ]:
# Cell 08 - Partition nodes by sign of the Fiedler vector

p1 = np.where(f >= 0)[0]  # nodes with non-negative Fiedler entries
p2 = np.where(f < 0)[0]  # nodes with negative Fiedler entries

display(as_latex(p1, prefix=r"\mathbf{P1}="))
display(as_latex(p2, prefix=r"\mathbf{P2}="))